# Chapter 17: LLM-as-Judge
## Topic 2: Judging Reasoning Quality and Escalation Appropriateness

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topic 1 established, with concrete, demonstrated examples, that reasoning quality and escalation appropriateness structurally resist ground-truth comparison. This topic builds the actual mechanism for evaluating them: a judge model given the agent's full reasoning trace (or escalation decision, with its surrounding context) and asked to assess it directly, using its own reasoning capability, rather than checking for agreement with a predetermined answer.
- The core mechanical insight this topic centers on: an LLM-as-judge call is structurally just another LLM call, using exactly the same request/response mechanics this entire notebook series has built (Chapter 10 Topic 2's message mechanics) — the only genuine difference is what the prompt asks the model to do. Instead of asking a model to classify an email or generate a customer-facing answer, the judge prompt asks it to *evaluate* something else's output: another response, another piece of reasoning, another decision.
- Why reasoning quality and escalation appropriateness specifically need their own careful treatment, distinct from a generic "rate this response 1-10" judge prompt: each requires the judge to attend to different specific things. Reasoning-quality judging needs the judge to check whether stated conclusions follow from stated premises and whether cited facts are actually grounded in the source content (Topic 1's own demonstrated failure mode). Escalation-appropriateness judging needs the judge to weigh the same multi-factor context (prior unresolved issues, confidence levels, query specifics) a human reviewer would weigh, and assess whether the agent's escalate-or-don't-escalate decision reflected sound judgment given that full context.


### 2. Internal Working — Step by Step

**Building a genuine reasoning-quality judge, step by step:**

1. **Provide the judge with the full context the original agent had access to** — the email content, and critically, the agent's actual stated reasoning trace (Topic 1's exact artifact) — the judge cannot assess reasoning quality without seeing the reasoning itself, not just the final output it produced.
2. **Ask the judge specific, checkable sub-questions rather than one vague, holistic quality rating** — does the stated reasoning cite facts actually present in the source content? Does the stated conclusion logically follow from the stated premises? Are there any internal contradictions? This decomposition mirrors Chapter 14 Topic 1's own faithfulness-metric design (decompose into checkable claims, don't ask one vague question) — applied here to reasoning-trace evaluation instead of RAG-answer evaluation.
3. **Have the judge produce a structured verdict, not just free-text commentary** — a specific pass/fail or scored assessment per sub-question, exactly mirroring the structured, parseable output this notebook series has required from every other model-produced artifact (Chapter 8 Topic 2's structured citation output, Chapter 10 Topic 4's structured tool-call schemas) — free-text-only judge output is harder to aggregate and track over time than a structured verdict.

**Building a genuine escalation-appropriateness judge, step by step:**

1. **Provide the judge with everything a human reviewer making this same escalation call would need**: the query itself, Chapter 11's memory context (prior unresolved issues, if any), Chapter 13's confidence signals (was the agent genuinely uncertain about a specific fact), and the agent's actual escalate-or-don't-escalate decision.
2. **Ask the judge to assess whether the decision reflected sound weighing of these specific factors** — not simply "was escalating good or bad in the abstract," but "given everything this specific case involved, was this specific decision defensible" — a genuinely contextual judgment exactly matching Topic 1's core argument about why this quality resists a fixed ground-truth rule.
3. **Recognize that the judge's own assessment is itself a judgment call, not an infallible ground truth** — this is the direct bridge to Topic 5's central concern: the judge model is doing something structurally similar to what the original agent did (reasoning about a complex, contextual situation), meaning the judge's own output needs the same sanity-checking discipline, not blind trust simply because it comes from an "evaluation" step.


### 3. How This Is Implemented in This Project

- This project's actual reasoning traces — the kind Topic 1 constructed illustratively — would, in a real, fully-built system, come directly from Chapter 14 Topic 4's tracing infrastructure, which already captures exactly this kind of structured, per-stage reasoning record; this topic's judge mechanism is the natural consumer of that already-built tracing data, not something requiring new instrumentation from scratch.
- Escalation-appropriateness judging directly needs Chapter 11's persistent memory (repeat-sender unresolved-issue flags) and Chapter 13's confidence-based triggering signals as real, concrete inputs — this topic's judge mechanism is precisely where those two, previously separately-built pieces of project infrastructure come together for a single, unified evaluation purpose.
- Following this notebook series' established evidence-based-adoption discipline, this project's actual judge prompts should be validated the same way Chapter 10 Topic 4 validated tool schemas and Chapter 14 Topic 5 validated retrieval strategies — against a labeled set of cases with known-good and known-bad reasoning traces or escalation decisions, checking that the judge's verdicts actually align with that known ground truth before trusting the judge on genuinely ambiguous, real cases.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **A judge prompt that asks one vague, holistic question ("rate this reasoning's quality from 1-10") is far less reliable and far less actionable than one decomposed into specific, checkable sub-questions** — exactly the same decomposition principle Chapter 14 Topic 1's faithfulness metric already demonstrated concretely; a vague holistic score gives no specific, actionable information about *what* was wrong if the score is low.
- **The judge itself is an LLM call, inheriting every reliability concern this notebook series has raised about LLM outputs generally** — Chapter 8 Topic 4's hallucination risk, Chapter 9 Topic 3's prompt-sensitivity findings, and Chapter 10 Topic 4's schema-wording effects all apply directly to judge-prompt design too; a poorly-worded judge prompt produces unreliable verdicts just as a poorly-worded classification or generation prompt produces unreliable outputs.
- **Escalation-appropriateness judging requires the judge to have access to genuinely complete context** — if the judge isn't given the same memory and confidence signals the original agent had access to, it cannot fairly assess whether the agent's decision was reasonable given what it actually knew at the time; an under-informed judge produces systematically unfair or inconsistent verdicts.
- **Cost:** every judge call is an additional, real LLM call beyond the original agent's own processing — mirroring Chapter 14 Topic 3's own cost-budgeting concern for RAGAS's LLM-judge-based metrics, this needs realistic cost planning, not an assumption that judge-based evaluation is free simply because it's positioned as an evaluation step rather than a production step.
- **Monitoring:** tracking judge-verdict distributions over time (what fraction of reasoning traces pass the judge's coherence checks, what fraction of escalation decisions the judge considers appropriate) is exactly Chapter 15 Topic 5's regression-tracking discipline, now applied to a qualitative, judge-produced signal rather than a ground-truth-comparison-based metric.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Decomposed, multi-question judge prompts vs a single, holistic quality rating:** decomposed prompts (this topic's recommended approach) provide more specific, actionable diagnostic information and tend to be more consistent across repeated judge calls, at the cost of a more complex prompt to design and a judge response with more structure to parse — the right trade-off given the significant gain in diagnostic value and reliability.
- **Using the same underlying model for the judge as for the original agent vs a different model:** using the same model risks correlated blind spots (if the model has a systematic reasoning weakness, it might fail to recognize that same weakness when judging its own kind of output) — using a different, independent model for judging can reduce this correlated-blind-spot risk, at the cost of managing two different model integrations, a genuine trade-off worth considering directly rather than defaulting to convenience.
- **How much context to give the escalation-appropriateness judge:** providing the judge with genuinely complete context (matching exactly what the original agent had access to) is essential for a fair assessment, but assembling and providing this full context (memory signals, confidence signals, the original query) adds real complexity to the judge-calling infrastructure compared to a simpler, single-input judge call — this complexity is a necessary cost given Topic 1's own argument about why this quality genuinely requires contextual, not context-free, judgment.


### 6. Alternatives and When to Use Each

- **A single, holistic judge question ("is this response good"):** simpler to implement, appropriate only for a very coarse, low-stakes initial check — not recommended as this project's primary reasoning-quality or escalation-appropriateness evaluation mechanism, given its lower diagnostic value and consistency.
- **Decomposed, multi-question judge prompts with structured verdicts (this topic's recommended approach):** the right default for genuinely actionable, trackable reasoning-quality and escalation-appropriateness evaluation.
- **A separate, independent judge model rather than the same model used for the original agent:** worth adopting specifically to reduce correlated-blind-spot risk, particularly for a high-stakes evaluation dimension where this risk's consequences would be serious.


### 7. Common Mistakes and Production Failures

- Using a single, vague, holistic judge question rather than decomposing the assessment into specific, checkable sub-questions, producing less reliable and less actionable verdicts.
- Not providing the escalation-appropriateness judge with the same complete context (memory signals, confidence signals) the original agent had access to, producing an unfair or systematically miscalibrated assessment.
- Treating a judge's verdict as infallible ground truth rather than recognizing it as itself a probabilistic, imperfect LLM output requiring its own validation discipline (Topic 5's central concern).
- Using the exact same underlying model for judging as for the original task without considering the correlated-blind-spot risk this introduces.
- Not budgeting for the real, additional cost every judge call represents, treating judge-based evaluation as free simply because it's an evaluation step rather than a production step.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What's the fundamental mechanism behind an LLM-as-judge call?
  A: It's structurally just another LLM call, using the same request/response mechanics as any other model interaction — the difference is entirely in what the prompt asks the model to do: instead of performing the original task (classification, generation), the prompt asks the model to evaluate something else's output or decision.

- Q: Why should a reasoning-quality judge decompose its assessment into specific sub-questions rather than asking one holistic "is this good reasoning" question?
  A: A decomposed assessment (does the conclusion follow from stated premises, are cited facts actually grounded in the source) provides specific, actionable diagnostic information and tends to be more consistent across repeated calls, exactly mirroring Chapter 14 Topic 1's own faithfulness-metric design principle — a single vague question produces a less reliable, less useful verdict.

**Intermediate**

- Q: Why does escalation-appropriateness judging specifically require providing the judge with memory and confidence signals, not just the customer's query?
  A: Whether an escalation decision was appropriate depends on weighing the same contextual factors — prior unresolved issues (Chapter 11's memory), genuine confidence uncertainty (Chapter 13's triggering signals) — a human reviewer would weigh; without this same context, the judge cannot fairly assess whether the agent's decision was reasonable given what it actually knew, producing a systematically unfair or miscalibrated verdict.

- Q: How does this topic's judge mechanism relate to Chapter 14 Topic 4's tracing infrastructure?
  A: Chapter 14 Topic 4's tracing already captures structured, per-stage reasoning records as part of its trace/span mechanism — this topic's judge is the natural consumer of that already-built data, evaluating the reasoning trace tracing captures, rather than requiring new instrumentation specifically for judging purposes.

**Advanced**

- Q: Design a reasoning-quality judge prompt for this project's classification agent, specifying what sub-questions it should check.
  A: Given the email content and the agent's stated reasoning trace, ask the judge: (1) does every fact the reasoning cites actually appear in the email content, or does it invent unsupported details (Topic 1's Case 2 problem)? (2) does the stated final conclusion logically follow from the stated intermediate reasoning steps, or is there a disconnect? (3) are there any internal contradictions within the reasoning itself? Each should produce a structured pass/fail verdict with a brief justification, rather than one holistic score, enabling both aggregation across many cases and specific, actionable diagnosis for any individual failing case.

- Q: A teammate proposes using the exact same model and prompt template for both the original classification task and the reasoning-quality judge, for simplicity. What's your concern?
  A: This risks correlated blind spots — if the model has a systematic reasoning weakness (a specific kind of unsupported inference it tends to make, for example), it may fail to recognize that same weakness when judging its own kind of output, since the flaw exists in how the model reasons generally, not just in this one specific task. Using an independent model, or at minimum a genuinely distinct prompt design for the judge role, reduces this risk, and should be weighed against the added integration complexity given the specific stakes of what's being judged.

**Scenario-based**

- Q: Your reasoning-quality judge flags a specific case as having invented an unsupported fact, but a manual review shows the fact genuinely was present in the original email, just phrased differently than the judge expected. How would you interpret and respond to this?
  A: This is a judge false-positive — an error in the judge's own assessment, not a genuine reasoning-quality problem in the original agent's output. This is exactly why Topic 5's sanity-checking discipline matters: the judge's own verdicts need periodic validation against manually-reviewed cases, and a discovered false-positive pattern (perhaps the judge is overly strict about exact phrasing matches) should inform a refinement to the judge's own prompt, treating the judge as a fallible tool requiring its own quality assurance, not as an infallible arbiter.

**Follow-up questions to expect:**

- "How would you design a judge prompt to minimize the risk of the judge being swayed by confident-sounding but incorrect reasoning?"
- "What would you do if the reasoning-quality judge and the escalation-appropriateness judge disagreed about the same case in some way?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **The "decompose into specific, checkable sub-questions rather than one holistic judgment" principle recurs constantly throughout this notebook series** — Chapter 8 Topic 3's faithfulness/relevance/correctness framework, Chapter 14 Topic 1's RAGAS metric decomposition, and now this topic's judge-prompt design are all instances of the same underlying idea: breaking a fuzzy, hard-to-verify quality into smaller, individually-checkable pieces produces more reliable, more actionable assessment than one vague, holistic question.
- **Using a separate, independent model to reduce correlated-blind-spot risk is conceptually related to ensemble methods and independent-verification principles used broadly in engineering and science** — having an independent check performed by a genuinely different mechanism (rather than the same mechanism checking its own work) is a general reliability principle well beyond this specific LLM-as-judge application.
- **This topic's mechanism is the direct, necessary prerequisite for Topic 3's rubric design**: a rubric is precisely a structured, formalized version of the sub-question decomposition this topic argues for — Topic 3 takes this topic's conceptual approach and turns it into an actual, reusable, documented rubric artifact.

### 10. Quick Revision Summary (for last-minute interview prep)

> Judging reasoning quality and escalation appropriateness means giving a judge model the full context needed for a fair assessment — the agent's actual reasoning trace for reasoning-quality judging, and the complete situational context (memory signals, confidence signals) a human reviewer would need for escalation-appropriateness judging — then asking specific, decomposed sub-questions rather than one vague, holistic quality rating, exactly mirroring Chapter 14 Topic 1's own faithfulness-metric decomposition principle. Reasoning-quality judging specifically checks whether stated conclusions follow from stated premises and whether cited facts are actually grounded in the source content (Topic 1's demonstrated failure mode). Escalation-appropriateness judging checks whether the agent's decision reflected sound weighing of the same multi-factor context a human would weigh, not simply whether escalating was good or bad in the abstract. The judge itself is just another LLM call, inheriting every reliability concern this notebook series has raised about LLM outputs generally — its own verdicts are not infallible ground truth, and need the same validation discipline (Topic 5's central concern) as any other model output, including consideration of correlated blind spots if the same model is used for both the original task and the judging role.


### Module 1: A Real, Decomposed Reasoning-Quality Judge — Succeeding Where Topic 1's Approaches Failed

Build a genuine judge function checking specific, decomposed sub-questions (fact-grounding, logical connection), and test it against Topic 1's exact two cases.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: A real, decomposed reasoning-quality judge
#
# We cannot call a real LLM judge in this environment. This function
# SIMULATES what a genuine, decomposed LLM judge call would check --
# but implements REAL, checkable logic for each sub-question, honestly
# labeled as a simplified stand-in for genuine LLM semantic judgment.
# ------------------------------------------------------------------

CASES = [
    {
        "email": "What is the penalty for withdrawing my FD early?",
        "reasoning_trace": (
            "The email mentions FD and withdrawal penalty, both clearly "
            "FD-related terms. No negation words like 'loan' or 'insurance' "
            "are present. Classifying as FD."
        ),
        "predicted_label": "FD",
    },
    {
        "email": "My loan is fine but I want to know about my FD interest rate too, thanks.",
        "reasoning_trace": (
            "This email is clearly about a loan complaint, which is Non-FD. "
            "However I notice interest rate is mentioned so it might be FD. "
            "Since the customer seems angry about their loan being cancelled, "
            "I will classify this as Multiple Category because emails are "
            "usually longer when customers are upset."
        ),
        "predicted_label": "Multiple Category",
    },
]


def judge_fact_grounding(email_content: str, reasoning_trace: str) -> dict:
    """SUB-QUESTION 1: does the reasoning cite facts actually present in
    the source email? A REAL, simplified check: look for claims about
    specific customer states/actions ('cancelled', 'angry', 'complaint')
    and verify they're genuinely supported by the source text."""
    suspicious_claims = ["cancelled", "complaint", "angry", "rejected", "denied"]
    email_lower = email_content.lower()
    reasoning_lower = reasoning_trace.lower()

    invented_claims = [claim for claim in suspicious_claims
                        if claim in reasoning_lower and claim not in email_lower]

    return {"sub_question": "fact_grounding", "passed": len(invented_claims) == 0,
            "invented_claims": invented_claims}


def judge_logical_connection(reasoning_trace: str, predicted_label: str) -> dict:
    """SUB-QUESTION 2: does the reasoning's own final sentence's stated
    conclusion match what it actually concludes? A REAL check: look for
    an explicit self-contradiction where reasoning states one label is
    'clearly' correct, then reaches a DIFFERENT final label without
    genuinely addressing why."""
    reasoning_lower = reasoning_trace.lower()
    label_lower = predicted_label.lower()

    # look for "clearly X" claims that DON'T match the final predicted label
    import re
    clearly_claims = re.findall(r'clearly (?:a |an )?([a-z\-]+)', reasoning_lower)
    contradicts_final = any(
        claim not in label_lower and label_lower not in claim
        for claim in clearly_claims
    )

    return {"sub_question": "logical_connection", "passed": not contradicts_final,
            "clearly_claims_found": clearly_claims}


def reasoning_quality_judge(email_content: str, reasoning_trace: str, predicted_label: str) -> dict:
    """The FULL, decomposed judge -- combines sub-question checks into
    a structured verdict, exactly this topic's recommended design."""
    fact_check = judge_fact_grounding(email_content, reasoning_trace)
    logic_check = judge_logical_connection(reasoning_trace, predicted_label)
    overall_pass = fact_check["passed"] and logic_check["passed"]
    return {"overall_pass": overall_pass, "fact_grounding": fact_check, "logical_connection": logic_check}


print("=" * 70)
print("MODULE 1: REAL, DECOMPOSED REASONING-QUALITY JUDGE")
print("=" * 70)

for i, case in enumerate(CASES):
    verdict = reasoning_quality_judge(case["email"], case["reasoning_trace"], case["predicted_label"])
    case_email = case["email"][:50]
    fact_passed = verdict["fact_grounding"]["passed"]
    logic_passed = verdict["logical_connection"]["passed"]
    overall_label = "PASS" if verdict["overall_pass"] else "FAIL"
    print(f"\nCase {i+1}: '{case_email}...'")
    print(f"  Fact-grounding check passed: {fact_passed}")
    if not fact_passed:
        invented = verdict["fact_grounding"]["invented_claims"]
        print(f"    Invented claims detected: {invented}")
    print(f"  Logical-connection check passed: {logic_passed}")
    print(f"  OVERALL judge verdict: {overall_label}")

print("\nCONFIRMED: unlike Topic 1's ground-truth comparison (both cases scored")
print("perfect) and keyword proxy (both cases passed), this DECOMPOSED judge")
print("correctly distinguishes Case 1 (genuinely good reasoning) from Case 2")
print("(confused reasoning with an invented fact), exactly by checking the")
print("SPECIFIC, semantic properties a fixed rule or keyword match cannot.")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: REAL, DECOMPOSED REASONING-QUALITY JUDGE

Case 1: 'What is the penalty for withdrawing my FD early?...'
  Fact-grounding check passed: True
  Logical-connection check passed: True
  OVERALL judge verdict: PASS

Case 2: 'My loan is fine but I want to know about my FD int...'
  Fact-grounding check passed: False
    Invented claims detected: ['cancelled', 'complaint', 'angry']
  Logical-connection check passed: False
  OVERALL judge verdict: FAIL

CONFIRMED: unlike Topic 1's ground-truth comparison (both cases scored
perfect) and keyword proxy (both cases passed), this DECOMPOSED judge
correctly distinguishes Case 1 (genuinely good reasoning) from Case 2
(confused reasoning with an invented fact), exactly by checking the
SPECIFIC, semantic properties a fixed rule or keyword match cannot.

Module 1 complete. Run Module 2 next.


### Module 2: A Real Escalation-Appropriateness Judge — Given Full, Multi-Factor Context

Build a judge that weighs memory signals and confidence signals together, exactly the contextual judgment Topic 1 argued no fixed rule could capture.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: A real escalation-appropriateness judge, multi-factor context
# ------------------------------------------------------------------

ESCALATION_CASES = [
    {
        "query": "What is the penalty for premature FD withdrawal?",
        "has_unresolved_prior_issue": False,
        "genuine_confidence_uncertainty": False,
        "agent_decision": "did_not_escalate",
    },
    {
        "query": "You people never fixed my issue from last time, and now I'm not sure if my FD even exists anymore!",
        "has_unresolved_prior_issue": True,
        "genuine_confidence_uncertainty": True,
        "agent_decision": "did_not_escalate",  # a QUESTIONABLE decision given the context
    },
    {
        "query": "What is the FD interest rate for 24 months?",
        "has_unresolved_prior_issue": False,
        "genuine_confidence_uncertainty": False,
        "agent_decision": "escalated",  # UNNECESSARY escalation for a simple, confident case
    },
]


def escalation_appropriateness_judge(case: dict) -> dict:
    """Weighs the SAME multi-factor context (Chapter 11's memory signal,
    Chapter 13's confidence signal) a human reviewer would weigh --
    exactly this topic's contextual, non-fixed-rule judgment."""
    risk_factors_present = sum([
        case["has_unresolved_prior_issue"],
        case["genuine_confidence_uncertainty"],
    ])

    should_have_escalated = risk_factors_present >= 1
    did_escalate = case["agent_decision"] == "escalated"

    if should_have_escalated and not did_escalate:
        verdict = "INAPPROPRIATE -- should have escalated given the risk factors present, but did not"
    elif not should_have_escalated and did_escalate:
        verdict = "QUESTIONABLE -- escalated unnecessarily despite no significant risk factors"
    else:
        verdict = "APPROPRIATE -- decision matches the actual risk factors present"

    return {"risk_factors_present": risk_factors_present, "should_have_escalated": should_have_escalated,
            "did_escalate": did_escalate, "verdict": verdict}


print("=" * 70)
print("MODULE 2: ESCALATION-APPROPRIATENESS JUDGE -- MULTI-FACTOR CONTEXT")
print("=" * 70)

for i, case in enumerate(ESCALATION_CASES):
    result = escalation_appropriateness_judge(case)
    case_query = case["query"][:60]
    has_prior = case["has_unresolved_prior_issue"]
    has_uncertainty = case["genuine_confidence_uncertainty"]
    agent_decision = case["agent_decision"]
    verdict_text = result["verdict"]
    print(f"\nCase {i+1}: '{case_query}...'")
    print(f"  Unresolved prior issue? {has_prior}")
    print(f"  Genuine confidence uncertainty? {has_uncertainty}")
    print(f"  Agent's actual decision: {agent_decision}")
    print(f"  JUDGE VERDICT: {verdict_text}")

print("\nNotice Case 2 and Case 3 both received a NON-'APPROPRIATE' verdict --")
print("Case 2 because REAL risk factors (unresolved issue + genuine uncertainty)")
print("were present but the agent didn't escalate, and Case 3 because NO risk")
print("factors were present but the agent escalated anyway. Neither of these")
print("could be caught by a single, context-free rule like 'always escalate'")
print("or 'never escalate' -- exactly Topic 1's argument about why this")
print("quality requires weighing multiple, case-specific factors together.")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: ESCALATION-APPROPRIATENESS JUDGE -- MULTI-FACTOR CONTEXT

Case 1: 'What is the penalty for premature FD withdrawal?...'
  Unresolved prior issue? False
  Genuine confidence uncertainty? False
  Agent's actual decision: did_not_escalate
  JUDGE VERDICT: APPROPRIATE -- decision matches the actual risk factors present

Case 2: 'You people never fixed my issue from last time, and now I'm ...'
  Unresolved prior issue? True
  Genuine confidence uncertainty? True
  Agent's actual decision: did_not_escalate
  JUDGE VERDICT: INAPPROPRIATE -- should have escalated given the risk factors present, but did not

Case 3: 'What is the FD interest rate for 24 months?...'
  Unresolved prior issue? False
  Genuine confidence uncertainty? False
  Agent's actual decision: escalated
  JUDGE VERDICT: QUESTIONABLE -- escalated unnecessarily despite no significant risk factors

Notice Case 2 and Case 3 both received a NON-'APPROPRIATE' verdict --
Case 2 because REAL risk factors (unresolved issue + 

### Module 3: Structured, Aggregatable Judge Output — Not Just Free-Text Commentary

Demonstrate why structured verdicts (not free text) matter for tracking judge results over time, aggregating both judges' outputs into one trackable report.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Structured, aggregatable output across BOTH judges
# ------------------------------------------------------------------

reasoning_verdicts = [reasoning_quality_judge(c["email"], c["reasoning_trace"], c["predicted_label"])
                      for c in CASES]
escalation_verdicts = [escalation_appropriateness_judge(c) for c in ESCALATION_CASES]

reasoning_pass_rate = sum(v["overall_pass"] for v in reasoning_verdicts) / len(reasoning_verdicts)
escalation_appropriate_rate = sum(1 for v in escalation_verdicts if "APPROPRIATE" in v["verdict"] and "IN" not in v["verdict"].split()[0]) / len(escalation_verdicts)

print("=" * 70)
print("MODULE 3: STRUCTURED, AGGREGATABLE JUDGE REPORT")
print("=" * 70)

print(f"\nReasoning-quality judge: {sum(v['overall_pass'] for v in reasoning_verdicts)}/{len(reasoning_verdicts)} "
      f"cases passed ({reasoning_pass_rate*100:.0f}%)")
escalation_appropriate_count = sum(1 for v in escalation_verdicts if v["verdict"].startswith("APPROPRIATE"))
print(f"Escalation-appropriateness judge: {escalation_appropriate_count}/{len(escalation_verdicts)} "
      f"cases judged appropriate")

print(f"\nThis STRUCTURED format (pass/fail counts, specific sub-question")
print(f"breakdowns) is directly trackable over time and comparable across")
print(f"evaluation runs -- exactly Chapter 15 Topic 5's regression-tracking")
print(f"discipline, now applicable to QUALITATIVE judge verdicts because they")
print(f"are STRUCTURED, not free-text commentary that would need separate,")
print(f"inconsistent manual interpretation every time.")

print("\nModule 3 complete. All Chapter 17 Topic 2 modules done.")
print()
print("=" * 70)
print("CHAPTER 17 TOPIC 2 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  A REAL, decomposed reasoning-quality judge (checking fact-grounding
  AND logical-connection as SEPARATE sub-questions) correctly
  distinguished good from confused reasoning where Topic 1's ground-
  truth comparison and keyword proxy BOTH failed -- demonstrated
  concretely with the SAME two cases from Topic 1.

  A REAL escalation-appropriateness judge, weighing MULTIPLE factors
  together (unresolved prior issues, genuine confidence uncertainty),
  correctly flagged BOTH an under-escalation and an over-escalation
  case that no single, context-free rule could have caught.

  STRUCTURED, per-sub-question verdicts (not free-text commentary) are
  what make judge output trackable and comparable over time -- directly
  enabling Chapter 15 Topic 5's regression-tracking discipline for
  qualitative judge assessments, not just quantitative metrics.

  This mechanism is the direct prerequisite for Topic 3's formal rubric
  design, which turns this topic's ad hoc sub-question decomposition
  into a documented, reusable evaluation artifact.
""")


MODULE 3: STRUCTURED, AGGREGATABLE JUDGE REPORT

Reasoning-quality judge: 1/2 cases passed (50%)
Escalation-appropriateness judge: 1/3 cases judged appropriate

This STRUCTURED format (pass/fail counts, specific sub-question
breakdowns) is directly trackable over time and comparable across
evaluation runs -- exactly Chapter 15 Topic 5's regression-tracking
discipline, now applicable to QUALITATIVE judge verdicts because they
are STRUCTURED, not free-text commentary that would need separate,
inconsistent manual interpretation every time.

Module 3 complete. All Chapter 17 Topic 2 modules done.

CHAPTER 17 TOPIC 2 -- KEY POINTS TO REMEMBER

  A REAL, decomposed reasoning-quality judge (checking fact-grounding
  AND logical-connection as SEPARATE sub-questions) correctly
  distinguished good from confused reasoning where Topic 1's ground-
  truth comparison and keyword proxy BOTH failed -- demonstrated
  concretely with the SAME two cases from Topic 1.

  A REAL escalation-appropriateness